2026.10.28 一致性修订：请 Restart Kernel & Run All。初始化会验证当前 embedding→特征→预测→概率；旧 niche 缓存不再复用，旧图输出已清空。Deployment 最佳模型用法保持不变。


# HCC unlabeled rest — pan-cancer Hist2Pheno

8 CODEX HCC **rest** regions without a CODEX cell-type CSV. Not the Visium `External_val/HCC` JPEGs. Notebook draws the **5 largest**. No TLS / SRI / TNI.

Checkpoint: `data/PanCancer/result_all_spatial/cross_dataset_cv/D_emph_L2_spatial_pancancer/best_mlp_gpu.pt`

```bash
CUDA_DEVICE=0 bash code/External_val/HCC_unlabel/demo.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
# 2026.10.28, explicit checkout override / Farm 可设 HIST2PHENO_REPO
import os
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "External_val"))

STAR_PROB_MIN = 0.0  # 2026.10.28, set above 0 to store a separate _probXX folder
import os as _prob_os
_prob_os.environ["STAR_PROB_MIN"] = "0" if float(STAR_PROB_MIN) <= 0 else f"{float(STAR_PROB_MIN):.2f}"
import unlabeled_external as ue

COHORT = "HCC_unlabel"
SAVE_FIGURES = False
FIGSIZE = (17.2, 12.5)
DPI = 160
SAVE_DPI = 300
HE_MAX_SIDE = 1400
MARKER_PX = 1.0

cov = ue.coverage(COHORT)
samples = ue.viz_samples(COHORT)
print(f"{COHORT}: {len(cov)} unlabeled slides; drawing {len(samples)}")
print("viz", samples)
display(cov.loc[cov["sample"].isin(samples)])

# 2026.10.28, mandatory current-input preflight / 后续直接读 h5ad 的图表也不得绕过上游
import os as _pipeline_os
_pipeline_os.environ["STAR_PROB_MIN"] = str(globals().get("STAR_PROB_MIN", 0.0))
if "ensure_l2_probs" in globals():
    _owner = ensure_l2_probs.__self__
    _current_samples = list(samples) if "samples" in globals() else _owner.discover_samples()
    _owner.cmd_predict(_current_samples, force=False, allow_cpu=True)
elif "ue" in globals():
    ue.cmd_predict(COHORT, samples, cuda_device=_pipeline_os.environ.get("CUDA_DEVICE", "0"),
                   force=False, allow_cpu=True, embedder=EMBEDDER)
else:
    raise RuntimeError("No current prediction validator; run the setup cell first")


In [ ]:
display(Markdown("## Predicted L1 / L12 / L2 (row 1 HE, row 2 points)"))
for sample in samples:
    pred = ue.pred_h5ad(ue._cohort(COHORT), sample)
    if not pred.is_file():
        print("missing pred", sample, "— run demo.sh predict")
        continue
    figs = ue.plot_spatial(
        COHORT,
        [sample],
        save=SAVE_FIGURES,
        figsize=FIGSIZE,
        dpi=DPI,
        save_dpi=SAVE_DPI,
        he_max_side=HE_MAX_SIDE,
        marker_px=MARKER_PX,
    )
    for fig in figs:
        display(fig)
        plt.close(fig)


In [ ]:
display(Markdown("## Predicted composition (no GT)"))
have = [s for s in samples if ue.pred_h5ad(ue._cohort(COHORT), s).is_file()]
if have:
    figs = ue.plot_composition(COHORT, have, save=SAVE_FIGURES, dpi=DPI, save_dpi=SAVE_DPI)
    for fig in figs:
        display(fig)
        plt.close(fig)
